# Qwen3 UltraChat-50k Results

This notebook reads saved training checkpoints and speculative-decoding eval summaries for the one-epoch UltraChat-50k Qwen3 sweeps. It does not load models, run training, or run evaluation.

Covered experiment pairs:

- `Qwen/Qwen3-14B` target with `Qwen/Qwen3-0.6B` draft
- `Qwen/Qwen3-8B` target with `Qwen/Qwen3-0.6B` draft

Expected artifacts are `checkpoints/<train_run>/meta.json`, optional HF trainer-state JSON files, and `/scratch/cs552-results/<eval_run>/eval_summary.json`.

## 1. Locate Artifacts

The defaults match the RunAI storage contract. Override `RESULTS_ROOT` or `CHECKPOINT_ROOT` in this cell if your artifacts live elsewhere.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = print

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO)

CHECKPOINT_ROOT = Path(os.environ.get("KDSD_CHECKPOINT_ROOT", REPO / "checkpoints")).expanduser()
RESULTS_ROOT = Path(os.environ.get("KDSD_RESULTS_ROOT", "/scratch/cs552-results")).expanduser()

DATA_ID = "ultrachat_50k"
SEED = 42
LOSSES = ("fkl", "rkl", "jsd", "ce")
EVAL_GAMMA = 4
EVAL_MAX_NEW_TOKENS = 256

print("repo root       :", REPO)
print("checkpoint root :", CHECKPOINT_ROOT)
print("results root    :", RESULTS_ROOT)
print("data id         :", DATA_ID)
print("seed            :", SEED)

## 2. Experiment Set

These run-name patterns are the ones produced by `scripts/run_qwen3_loss_sweep.sh`, `scripts/submit_qwen3_loss_sweep.sh`, and `scripts/submit_qwen3_8b_loss_sweep.sh`.

In [ ]:
EXPERIMENTS = [
    {
        "target_group": "Qwen3-14B target",
        "target": "Qwen/Qwen3-14B",
        "draft": "Qwen/Qwen3-0.6B",
        "train_prefix": "qwen3_0p6b",
        "eval_prefix": "qwen3_0p6b",
    },
    {
        "target_group": "Qwen3-8B target",
        "target": "Qwen/Qwen3-8B",
        "draft": "Qwen/Qwen3-0.6B",
        "train_prefix": "qwen3_8btarget_0p6b",
        "eval_prefix": "qwen3_8btarget_0p6b",
    },
]


def train_run_name(exp: dict[str, str], loss: str) -> str:
    return f"{exp['train_prefix']}_{loss}_{DATA_ID}_seed{SEED}"


def eval_run_name(exp: dict[str, str], loss_or_pretrain: str) -> str:
    return (
        f"{exp['eval_prefix']}_{loss_or_pretrain}_{DATA_ID}_seed{SEED}"
        f"_eval_g{EVAL_GAMMA}_max{EVAL_MAX_NEW_TOKENS}"
    )


for exp in EXPERIMENTS:
    print(exp["target_group"])
    print("  pretrained eval:", eval_run_name(exp, "pretrain"))
    for loss in LOSSES:
        print(" ", loss, "train:", train_run_name(exp, loss))
        print(" ", loss, "eval :", eval_run_name(exp, loss))

## 3. Load Helpers

In [ ]:
def load_json(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        with path.open("r", encoding="utf-8") as fh:
            value = json.load(fh)
    except Exception as exc:
        return {"_load_error": f"{type(exc).__name__}: {exc}"}
    return value if isinstance(value, dict) else {"_load_error": "JSON root is not an object"}


def is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value))


def fmt_number(value: Any, digits: int = 4) -> str:
    if value is None or value == "":
        return ""
    if is_number(value):
        return f"{float(value):.{digits}g}"
    return str(value)


def checkpoint_sort_key(path: Path) -> tuple[int, float]:
    match = re.search(r"checkpoint-(\d+)", str(path))
    step = int(match.group(1)) if match else -1
    try:
        mtime = path.stat().st_mtime
    except OSError:
        mtime = 0.0
    return step, mtime


def trainer_state_candidates(run_dir: Path) -> list[Path]:
    base = run_dir / "trainer_state"
    candidates = [base / "trainer_state.json"]
    if base.exists():
        candidates.extend(sorted(base.glob("checkpoint-*/trainer_state.json"), key=checkpoint_sort_key))
    return [path for path in candidates if path.exists()]


def load_latest_trainer_state(run_dir: Path) -> tuple[dict[str, Any] | None, Path | None]:
    candidates = trainer_state_candidates(run_dir)
    if not candidates:
        return None, None
    direct = run_dir / "trainer_state" / "trainer_state.json"
    path = direct if direct in candidates else candidates[-1]
    return load_json(path), path


def last_log_value(log_history: list[dict[str, Any]], key: str) -> Any:
    for record in reversed(log_history):
        if key in record and record[key] is not None:
            return record[key]
    return None


def first_existing_path(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def flatten_quality(summary: dict[str, Any] | None) -> dict[str, Any]:
    if not summary:
        return {}
    quality = summary.get("quality_score", {})
    if not isinstance(quality, dict):
        return {}
    return {f"quality:{key}": value for key, value in quality.items()}


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    if not rows:
        return "No rows."
    rendered = [[fmt_number(row.get(col)) for col in columns] for row in rows]
    widths = [len(col) for col in columns]
    for row in rendered:
        widths = [max(width, len(cell)) for width, cell in zip(widths, row)]
    header = "| " + " | ".join(col.ljust(width) for col, width in zip(columns, widths)) + " |"
    sep = "| " + " | ".join("-" * width for width in widths) + " |"
    body = ["| " + " | ".join(cell.ljust(width) for cell, width in zip(row, widths)) + " |" for row in rendered]
    return "\n".join([header, sep, *body])


def show_table(rows: list[dict[str, Any]], columns: list[str], *, title: str) -> None:
    print(title)
    if pd is not None:
        frame = pd.DataFrame(rows)
        for col in columns:
            if col not in frame.columns:
                frame[col] = None
        display(frame[columns])
    else:
        text = markdown_table(rows, columns)
        if Markdown is not None:
            display(Markdown(text))
        else:
            print(text)

## 4. Training Metrics

In [ ]:
def training_row(exp: dict[str, str], loss: str) -> dict[str, Any]:
    run = train_run_name(exp, loss)
    run_dir = CHECKPOINT_ROOT / run
    meta_path = run_dir / "meta.json"
    config_path = run_dir / "config.yaml"
    model_dir = run_dir / "model"

    meta = load_json(meta_path)
    state, state_path = load_latest_trainer_state(run_dir)
    log_history = state.get("log_history", []) if isinstance(state, dict) else []
    if not isinstance(log_history, list):
        log_history = []

    status_bits = []
    if not meta_path.exists():
        status_bits.append("missing meta")
    if not model_dir.exists():
        status_bits.append("missing model")
    if state_path is None:
        status_bits.append("missing trainer_state")
    if isinstance(meta, dict) and meta.get("_load_error"):
        status_bits.append("meta load error")
    if isinstance(state, dict) and state.get("_load_error"):
        status_bits.append("trainer_state load error")

    meta = meta if isinstance(meta, dict) and "_load_error" not in meta else {}
    state = state if isinstance(state, dict) and "_load_error" not in state else {}

    return {
        "target_group": exp["target_group"],
        "target": meta.get("target", exp["target"]),
        "draft": meta.get("draft_init", exp["draft"]),
        "loss": loss,
        "train_run": run,
        "status": "; ".join(status_bits) if status_bits else "ok",
        "steps": meta.get("steps", state.get("global_step")),
        "epoch": last_log_value(log_history, "epoch"),
        "train_loss_final": meta.get("train_loss_final"),
        "last_logged_loss": last_log_value(log_history, "loss"),
        "last_eval_loss": last_log_value(log_history, "eval_loss"),
        "last_loss_ce": last_log_value(log_history, "loss_ce"),
        "last_loss_kd": last_log_value(log_history, "loss_kd"),
        "learning_rate": last_log_value(log_history, "learning_rate"),
        "checkpoint_dir": str(run_dir),
        "trainer_state": str(state_path) if state_path else "",
        "config_exists": config_path.exists(),
    }


training_rows = [training_row(exp, loss) for exp in EXPERIMENTS for loss in LOSSES]

training_columns = [
    "target_group",
    "loss",
    "status",
    "steps",
    "epoch",
    "train_loss_final",
    "last_logged_loss",
    "last_eval_loss",
    "last_loss_ce",
    "last_loss_kd",
    "learning_rate",
    "train_run",
]

show_table(training_rows, training_columns, title="Training summary")

## 5. Speculative-Decoding Eval Metrics

In [ ]:
def eval_row(exp: dict[str, str], loss_or_pretrain: str) -> dict[str, Any]:
    run = eval_run_name(exp, loss_or_pretrain)
    result_dir = RESULTS_ROOT / run
    summary_path = result_dir / "eval_summary.json"
    timing_path = result_dir / "timing.json"
    config_path = result_dir / "config.yaml"
    generations_path = result_dir / "generations.jsonl"

    summary = load_json(summary_path)
    status_bits = []
    if not summary_path.exists():
        status_bits.append("missing eval_summary")
    if not timing_path.exists():
        status_bits.append("missing timing")
    if isinstance(summary, dict) and summary.get("_load_error"):
        status_bits.append("summary load error")

    summary = summary if isinstance(summary, dict) and "_load_error" not in summary else {}
    decoding = summary.get("decoding", {}) if isinstance(summary.get("decoding", {}), dict) else {}
    engines = summary.get("engines", {}) if isinstance(summary.get("engines", {}), dict) else {}
    hf_engine = engines.get("hf", {}) if isinstance(engines.get("hf", {}), dict) else {}

    row = {
        "target_group": exp["target_group"],
        "target": summary.get("target", exp["target"]),
        "draft": summary.get("draft", exp["draft"]),
        "loss": loss_or_pretrain,
        "run_kind": "pretrained" if loss_or_pretrain == "pretrain" else "trained",
        "eval_run": run,
        "status": "; ".join(status_bits) if status_bits else "ok",
        "acceptance_rate": summary.get("acceptance_rate"),
        "avg_accepted_tokens": summary.get("avg_accepted_tokens"),
        "speedup": summary.get("speedup"),
        "tokens_per_second": summary.get("tokens_per_second"),
        "sd_time_s": summary.get("sd_time_s"),
        "vanilla_time_s": summary.get("vanilla_time_s"),
        "n_prompts": summary.get("n_prompts"),
        "n_warmup": summary.get("n_warmup"),
        "n_repeats": summary.get("n_repeats"),
        "gamma": decoding.get("num_assistant_tokens", decoding.get("gamma")),
        "max_new_tokens": decoding.get("max_new_tokens"),
        "decoding_mode": decoding.get("mode"),
        "n_outer_steps": hf_engine.get("n_outer_steps"),
        "target_calls": hf_engine.get("target_calls"),
        "draft_calls": hf_engine.get("draft_calls"),
        "draft_forward_s": hf_engine.get("draft_forward_s"),
        "target_forward_s": hf_engine.get("target_forward_s"),
        "result_dir": str(result_dir),
        "config_exists": config_path.exists(),
        "generations_exists": generations_path.exists(),
    }
    row.update(flatten_quality(summary))
    return row


eval_rows = []
for exp in EXPERIMENTS:
    eval_rows.append(eval_row(exp, "pretrain"))
    for loss in LOSSES:
        eval_rows.append(eval_row(exp, loss))


def add_baseline_deltas(rows: list[dict[str, Any]]) -> None:
    baselines = {
        row["target_group"]: row
        for row in rows
        if row.get("run_kind") == "pretrained" and row.get("status") == "ok"
    }
    for row in rows:
        base = baselines.get(row["target_group"])
        if not base or row.get("run_kind") == "pretrained":
            row["speedup_delta_vs_pretrain"] = None
            row["acceptance_delta_vs_pretrain"] = None
            continue
        row["speedup_delta_vs_pretrain"] = (
            row.get("speedup") - base.get("speedup")
            if is_number(row.get("speedup")) and is_number(base.get("speedup"))
            else None
        )
        row["acceptance_delta_vs_pretrain"] = (
            row.get("acceptance_rate") - base.get("acceptance_rate")
            if is_number(row.get("acceptance_rate")) and is_number(base.get("acceptance_rate"))
            else None
        )


def add_best_flags(rows: list[dict[str, Any]]) -> None:
    for target_group in sorted({row["target_group"] for row in rows}):
        group_rows = [row for row in rows if row["target_group"] == target_group and row.get("status") == "ok"]
        speedups = [float(row["speedup"]) for row in group_rows if is_number(row.get("speedup"))]
        accepts = [float(row["acceptance_rate"]) for row in group_rows if is_number(row.get("acceptance_rate"))]
        best_speedup = max(speedups) if speedups else None
        best_accept = max(accepts) if accepts else None
        for row in rows:
            if row["target_group"] != target_group:
                continue
            row["best_speedup_in_target_group"] = bool(
                best_speedup is not None and is_number(row.get("speedup")) and math.isclose(float(row["speedup"]), best_speedup)
            )
            row["best_acceptance_in_target_group"] = bool(
                best_accept is not None and is_number(row.get("acceptance_rate")) and math.isclose(float(row["acceptance_rate"]), best_accept)
            )


add_baseline_deltas(eval_rows)
add_best_flags(eval_rows)

quality_columns = sorted({key for row in eval_rows for key in row if key.startswith("quality:")})
eval_columns = [
    "target_group",
    "loss",
    "run_kind",
    "status",
    "acceptance_rate",
    "avg_accepted_tokens",
    "speedup",
    "speedup_delta_vs_pretrain",
    "acceptance_delta_vs_pretrain",
    "tokens_per_second",
    "sd_time_s",
    "vanilla_time_s",
    "n_prompts",
    "gamma",
    "max_new_tokens",
    "best_speedup_in_target_group",
    "best_acceptance_in_target_group",
    *quality_columns,
    "eval_run",
]

show_table(eval_rows, eval_columns, title="Speculative-decoding eval summary")

## 6. Cross-Target Comparison

This view aligns the same loss across the 14B-target and 8B-target sweeps.

In [ ]:
def row_by_target_and_loss(rows: list[dict[str, Any]], target_group: str, loss: str) -> dict[str, Any] | None:
    for row in rows:
        if row.get("target_group") == target_group and row.get("loss") == loss:
            return row
    return None


comparison_rows = []
for loss in ("pretrain", *LOSSES):
    row_14b = row_by_target_and_loss(eval_rows, "Qwen3-14B target", loss)
    row_8b = row_by_target_and_loss(eval_rows, "Qwen3-8B target", loss)
    comparison_rows.append(
        {
            "loss": loss,
            "14b_status": row_14b.get("status") if row_14b else "missing row",
            "8b_status": row_8b.get("status") if row_8b else "missing row",
            "14b_acceptance": row_14b.get("acceptance_rate") if row_14b else None,
            "8b_acceptance": row_8b.get("acceptance_rate") if row_8b else None,
            "acceptance_8b_minus_14b": (
                row_8b.get("acceptance_rate") - row_14b.get("acceptance_rate")
                if row_14b and row_8b and is_number(row_14b.get("acceptance_rate")) and is_number(row_8b.get("acceptance_rate"))
                else None
            ),
            "14b_speedup": row_14b.get("speedup") if row_14b else None,
            "8b_speedup": row_8b.get("speedup") if row_8b else None,
            "speedup_8b_minus_14b": (
                row_8b.get("speedup") - row_14b.get("speedup")
                if row_14b and row_8b and is_number(row_14b.get("speedup")) and is_number(row_8b.get("speedup"))
                else None
            ),
        }
    )

comparison_columns = [
    "loss",
    "14b_status",
    "8b_status",
    "14b_acceptance",
    "8b_acceptance",
    "acceptance_8b_minus_14b",
    "14b_speedup",
    "8b_speedup",
    "speedup_8b_minus_14b",
]

show_table(comparison_rows, comparison_columns, title="8B-target minus 14B-target comparison")

## 7. Conclusions From Loaded Results

In [ ]:
def valid_metric_rows(rows: list[dict[str, Any]], metric: str, *, target_group: str | None = None) -> list[dict[str, Any]]:
    out = []
    for row in rows:
        if target_group is not None and row.get("target_group") != target_group:
            continue
        if row.get("status") == "ok" and is_number(row.get(metric)):
            out.append(row)
    return out


def best_by_metric(rows: list[dict[str, Any]], metric: str, *, target_group: str, trained_only: bool = False) -> dict[str, Any] | None:
    candidates = valid_metric_rows(rows, metric, target_group=target_group)
    if trained_only:
        candidates = [row for row in candidates if row.get("run_kind") == "trained"]
    if not candidates:
        return None
    return max(candidates, key=lambda row: float(row[metric]))


def metric_phrase(row: dict[str, Any] | None, metric: str, suffix: str = "") -> str:
    if row is None:
        return "not available"
    value = fmt_number(row.get(metric), digits=4)
    return f"{row.get('loss')} ({value}{suffix})"


lines = ["### Conclusions", ""]

missing_eval = [row for row in eval_rows if row.get("status") != "ok"]
missing_train = [row for row in training_rows if row.get("status") != "ok"]
if missing_eval or missing_train:
    lines.append(
        f"- Some artifacts are missing or incomplete: {len(missing_train)} training rows and "
        f"{len(missing_eval)} eval rows are not fully available. Treat conclusions as partial."
    )

for target_group in [exp["target_group"] for exp in EXPERIMENTS]:
    best_speed = best_by_metric(eval_rows, "speedup", target_group=target_group)
    best_accept = best_by_metric(eval_rows, "acceptance_rate", target_group=target_group)
    best_trained_speed = best_by_metric(eval_rows, "speedup", target_group=target_group, trained_only=True)
    best_trained_accept = best_by_metric(eval_rows, "acceptance_rate", target_group=target_group, trained_only=True)
    lines.append(
        f"- {target_group}: best overall speedup is {metric_phrase(best_speed, 'speedup', 'x')}; "
        f"best trained-run speedup is {metric_phrase(best_trained_speed, 'speedup', 'x')}."
    )
    lines.append(
        f"- {target_group}: best overall acceptance is {metric_phrase(best_accept, 'acceptance_rate')}; "
        f"best trained-run acceptance is {metric_phrase(best_trained_accept, 'acceptance_rate')}."
    )

for target_group in [exp["target_group"] for exp in EXPERIMENTS]:
    ce = row_by_target_and_loss(eval_rows, target_group, "ce")
    pretrain = row_by_target_and_loss(eval_rows, target_group, "pretrain")
    kd_rows = [row_by_target_and_loss(eval_rows, target_group, loss) for loss in ("fkl", "rkl", "jsd")]
    kd_rows = [row for row in kd_rows if row and row.get("status") == "ok" and is_number(row.get("speedup"))]
    best_kd_speed = max(kd_rows, key=lambda row: float(row["speedup"])) if kd_rows else None
    if ce and ce.get("status") == "ok" and best_kd_speed and is_number(ce.get("speedup")):
        delta = float(best_kd_speed["speedup"]) - float(ce["speedup"])
        direction = "outperforms" if delta > 0 else "does not outperform"
        lines.append(
            f"- {target_group}: the best KD loss by speedup ({best_kd_speed['loss']}) {direction} CE "
            f"by {delta:.4g}x."
        )
    if pretrain and pretrain.get("status") == "ok" and best_kd_speed and is_number(pretrain.get("speedup")):
        delta = float(best_kd_speed["speedup"]) - float(pretrain["speedup"])
        direction = "improves on" if delta > 0 else "does not improve on"
        lines.append(
            f"- {target_group}: the best KD loss by speedup ({best_kd_speed['loss']}) {direction} the pretrained draft "
            f"baseline by {delta:.4g}x."
        )

paired_deltas = [row for row in comparison_rows if is_number(row.get("speedup_8b_minus_14b"))]
if paired_deltas:
    positives = [row for row in paired_deltas if float(row["speedup_8b_minus_14b"]) > 0]
    negatives = [row for row in paired_deltas if float(row["speedup_8b_minus_14b"]) < 0]
    if len(positives) > len(negatives):
        lines.append("- Across matched losses, the 8B-target distillation/eval setup more often has higher speedup than the 14B-target setup.")
    elif len(negatives) > len(positives):
        lines.append("- Across matched losses, the 14B-target distillation/eval setup more often has higher speedup than the 8B-target setup.")
    else:
        lines.append("- Across matched losses, 8B-target and 14B-target speedups are mixed; there is no clear winner by count.")

complete_eval_rows = [row for row in eval_rows if row.get("status") == "ok"]
prompt_counts = sorted({row.get("n_prompts") for row in complete_eval_rows if row.get("n_prompts") is not None})
decode_settings = sorted({(row.get("gamma"), row.get("max_new_tokens"), row.get("n_repeats")) for row in complete_eval_rows})
if len(prompt_counts) > 1:
    lines.append(f"- Caveat: eval prompt counts differ across available summaries: {prompt_counts}.")
if len(decode_settings) > 1:
    lines.append(f"- Caveat: decoding or repeat settings differ across summaries: {decode_settings}.")
if not complete_eval_rows:
    lines.append("- No complete eval summaries were found, so conclusions cannot be computed yet.")

conclusion_md = "\n".join(lines)
if Markdown is not None:
    display(Markdown(conclusion_md))
else:
    print(conclusion_md)

## 8. Artifact Paths

Use this cell when a row is marked missing.

In [ ]:
path_rows = []
for row in training_rows:
    path_rows.append(
        {
            "kind": "train",
            "target_group": row["target_group"],
            "loss": row["loss"],
            "status": row["status"],
            "path": row["checkpoint_dir"],
        }
    )
for row in eval_rows:
    path_rows.append(
        {
            "kind": "eval",
            "target_group": row["target_group"],
            "loss": row["loss"],
            "status": row["status"],
            "path": row["result_dir"],
        }
    )

show_table(path_rows, ["kind", "target_group", "loss", "status", "path"], title="Resolved artifact paths")